# Computed metrics

In [1]:
import pandas as pd

main_df = pd.read_csv("../data/good/master_data_12k.csv", usecols=["team", "global_id", "sequence"])

In [2]:
# NOTE needs missing boltz structures

df = pd.read_csv("../data/needs_recomputing/cpu_boltz_rosetta_stats.csv", usecols=["name", "B_hs_engaged", "C_hs_engaged", "all_hs_engaged", "iplddt", "pae", "pde"])
df["global_id"] = df["name"].apply(lambda x: int(x.split("_")[0].split("binder")[-1]))
df = df.drop(columns=["name"])
df.columns = ["boltz_B_hs_engaged", "boltz_C_hs_engaged", "boltz_all_hs_engaged", "boltz_hs_iplddt", "boltz_hs_pae", "boltz_hs_pde", "global_id"]

main_df = main_df.merge(df, on="global_id", how="left")

In [3]:
# # NOTE: CAD Pro had X's in their seqs, so merging by seq ignores these 32. Needs to be fixed
# # NOTE Jakub calculates these too; just using those
# df = pd.read_csv("../data/needs_recomputing/esm2_log_likelihood.csv", usecols=["sequences", "seq_log_likelihood", ])
# df = df.rename(columns={"seq_log_likelihood": "esm2_pll", "sequences": "sequence"})
# df["esm2_pll"] = df["esm2_pll"] / df["sequence"].str.len()
# # main_df[main_df.merge(df, on="sequence", how="left")["esm2_pll"].isna()]
# main_df = main_df.merge(df, on="sequence", how="left")

In [4]:
# NOTE needs missing boltz structures
df = pd.read_csv("../data/needs_recomputing/ipsae_stats.csv")
df["global_id"] = df["model"].apply(lambda x: int(x.split("_")[0].split("binder")[-1]))
df = df.drop(columns=["model"])
df = df.rename(columns={col: f"boltz_{col}" for col in df.columns if col != "global_id"})
main_df = main_df.merge(df, on="global_id", how="left")

In [5]:
# NOTE needs missing boltz structures
df = pd.read_csv("../data/needs_recomputing/thread_boltz_rosetta_stats.csv")
df["global_id"] = df["name"].apply(lambda x: int(x.split("_")[0].split("binder")[-1]))
df = df.drop(columns=["name", "boltz", "scores", "pdb", "name.1", "description"])
# prepend "boltz_rosetta_" in front of cols except global_id
df.columns = [f"boltz_rosetta_{col}" if col != "global_id" else col for col in df.columns]

main_df = main_df.merge(df, on="global_id", how="left")

In [6]:
# NOTE needs missing boltz structures
df = pd.read_csv("../data/needs_recomputing/axis_angles_aterm_vs_bc.csv", usecols=["file","angle_Cterm_deg","angle_Nterm_deg"])
df["global_id"] = df["file"].apply(lambda x: x.split("/")[-1].split("_")[0].split("binder")[-1]).astype(int)
df = df.sort_values(by="global_id")
df = df.drop(columns=["file"])
# prepend "boltz_" in front of cols except global_id
df.columns = [f"boltz_{col}" if col != "global_id" else col for col in df.columns]
main_df = main_df.merge(df, on="global_id", how="left")

In [7]:
# NOTE All good!
df = pd.read_csv("../data/good/mmseqs/all_vs_cd20ab.tsv", sep="\t", names=["query","target","pident","alnlen","qlen","tlen","qcov","tcov","evalue"])
df["global_id"] = df["query"].apply(lambda x: int(x.split("|")[0]))
df = df.drop(columns=["query", "alnlen", "qlen", "tlen", "qcov", "tcov", "evalue"])
df.columns = ["closest_ab", "closest_ab_pident", "global_id"]

main_df = main_df.merge(df, on="global_id", how="left")

In [8]:
# NOTE All good!
df = pd.read_csv("../data/good/mmseqs/all_vs_ubiquitin.tsv", sep="\t", names=["query","target","pident","alnlen","qlen","tlen","qcov","tcov","evalue"])
df["global_id"] = df["query"].apply(lambda x: int(x.split("|")[0]))
df = df.drop(columns=["query", "target", "alnlen", "qlen", "tlen", "qcov", "tcov", "evalue"])
df.columns = ["ubiquitin_pident", "global_id"]
main_df = main_df.merge(df, on="global_id", how="left")

In [9]:
# NOTE: Good!

df = pd.read_csv("../data/good/max_pairwise_identities_hamming.csv")
df = df.drop(columns=["team", "sequence"])
# append _hamming to all columns except global_id
df = df.rename(columns={col: f"{col}_hamming" for col in df.columns if col != "global_id"})
main_df = main_df.merge(df, on="global_id", how="left")

In [10]:
# NOTE: Good!

df = pd.read_csv("../data/good/max_pairwise_identities_mmseqs.csv")
df = df.drop(columns=["seq_group"])
# append _hamming to all columns except global_id
df = df.rename(columns={col: f"{col}_mmseqs" for col in df.columns if col != "global_id"})
main_df = main_df.merge(df, on="global_id", how="left")

In [11]:
# NOTE: Good!

df = pd.read_csv("../data/jakub/12k_all_jakub.csv")
for col in df.columns:
    if "esm2" in col:
        # normalize
        df[col] = df[col] / df["sequence_x"].str.len()
df = df.drop(columns=["sequence_x", "sequence_y"])

# in col names, rename "chai_" with "chai1_"
df = df.rename(columns={col: col.replace("chai_", "chai1_") for col in df.columns})


main_df = main_df.merge(df, on="global_id", how="left")

In [12]:
# NOTE: Good! Derived from Jakub's data
df = pd.read_csv("../data/jakub/disulfide_and_pipi_contacts.csv")
df = df.drop(columns=["pdb_path", "disulfide_within_chain", "disulfide_between_chain", "pipi_within_chain", "pipi_between_chain"])
main_df = main_df.merge(df, on="global_id", how="left")

In [13]:
# NOTE: Need to recompute for missing boltz structures
df = pd.read_csv("../data/needs_recomputing/phillip/binder_sapscore.csv")
df["global_id"] = df["binder"].apply(lambda x: int(x.split("binder")[-1]))
df = df.drop(columns=["binder"])
df.columns = ["boltz_sap_score", "global_id"]
main_df = main_df.merge(df, on="global_id", how="left")

In [14]:
# NOTE: Need to add in the missing boltz structures
df = pd.read_csv("../data/needs_recomputing/mpnn/consolidated_mpnn_scores_pivoted.csv")
# prepend "boltz_" to all columns except global_id
df.columns = [f"boltz_{col}" if col != "global_id" else col for col in df.columns]

main_df = main_df.merge(df, on="global_id", how="left")

In [ ]:
# NOTE: Good!
df = pd.read_csv("../data/good/leah_12k_dssp.csv")
main_df = main_df.merge(df, on="global_id", how="left")

,global_id,dssp
0,1,-EEE------BSSS---GGGS-EEEEEE-TT--EEEEEEE--TTT-...
1,2,-EEEEE----S--TTS-GGGS-EEEEEE-TT--EEEEEEE--STT-...
2,3,---------S---S---GGGSEEEEEEE-TT--EEEEEEE--STT-...
3,4,-EEEEE----SGGG-S-GGGS-EEEEEE-TT--EEEEEEE--TTT-...
4,5,------EEE--TT----GGGS-EEEEEE-TT--EEEEEEE--TTTT...
...,...,...
11995,11996,-HHHHHHHHHHHHHHHHH-HHHHHHHHHHHHHHSSSHHHHHHHHHH...
11996,11997,-HHHHHHHHHHHHHHHHHHT--HHHHHHHHHHHHHHHHHHHT-HHH...
11997,11998,-HHHHHHHHHHHHHHHHHHHHHHHHTT-HHHHHHHHHHHHHHHHHH...
11998,11999,---HHHHHHHHHHHHHHH-HHHHHHHHHHHHHHT--HHHHHHHHHH...


In [16]:
main_df.to_csv("../data/12k_all_metrics.csv", index=False)

# Experimental Results

In [17]:
import pandas as pd

results_df = pd.read_csv("../data/good/master_data_12k.csv", usecols=["global_id", "sequence"])

In [18]:
# NOTE: Good!
df = pd.read_csv("../data/good/leah_12k_comb_raw_read_counts.csv")
df["global_id"] = df["global_id"].astype(int)

# prepend with leah_12k_ to all columns except global_id
df = df.rename(columns={col: f"leah_12k_{col}" if col != "global_id" else col for col in df.columns})
results_df = results_df.merge(df, on="global_id", how="left")

In [19]:
# NOTE: Good!
df = pd.read_csv("../data/good/leah_12k_dna.csv")
df["global_id"] = df["global_id"].astype(int)
df = df[["global_id", "full_optimized_sequence"]]
df = df.rename(columns={"full_optimized_sequence": "dna_sequence"})
results_df = results_df.merge(df, on="global_id", how="left")

In [20]:
# NOTE: Good!
df = pd.read_csv("../data/good/Q-424341_leah_normalized_count_v2.csv")

results_df = results_df.merge(df, on="dna_sequence", how="left")

In [21]:
# NOTE: Good!
df = pd.read_csv("../data/good/leah_12k_edgeR_statistics.csv")
df = df.rename(columns={"SequenceID": "global_id"})
df["global_id"] = df["global_id"].astype(int)

# prepend with leah_12k_ to all columns except global_id
df = df.rename(columns={col: f"leah_12k_{col}" if col != "global_id" else col for col in df.columns})
df = df.rename(columns={"leah_12k_2-fold threshold?": "leah_12k_2fold_threshold", "leah_12k_Significant?": "leah_12k_significant"})

results_df = results_df.merge(df, on="global_id", how="left")

results_df["leah_12k_detected"] = ~results_df["leah_12k_2fold_threshold"].isna()

In [22]:
# NOTE: Good! Only the top 10 teams
df = pd.read_csv("../data/good/master_data_top10.csv")
df = df.iloc[2:].reset_index(drop=True) # remove + and - controls
df = df.drop(columns=["team", "Final Ranking"])

# global_id	Cytotoxicity AUC (R: CD20)	Cytotoxicity AUC (K: Control)	Fold Change in CTV MFI AUC (Proliferation)	Cytotoxicity AUC (R-K)	% Cytokine+	Fold Change in Total Cell Count/Well (Expansion)	Cytotoxicity AUC (R-K) MinMax Norm	% Cytokine+ MinMax Norm	Fold Change in Total Cell Count/Well (Expansion) MinMax Norm	Sum of Norms

df.columns = ["global_id", "cytotox_AUC_R_CD20", "cytotox_AUC_K_Control", "fold_change_CTV_MFI_AUC_proliferation", "cytotox_AUC_R_minus_K", "percent_cytokine_positive", "fold_change_total_cell_count_per_well_expansion", "cytotox_AUC_R_minus_K_minmax_norm", "percent_cytokine_positive_minmax_norm", "fold_change_total_cell_count_per_well_expansion_minmax_norm", "sum_of_norms"]
# add leah_top10_ prefix to all columns except global_id
df = df.rename(columns={col: f"leah_top10_{col}" if col != "global_id" else col for col in df.columns})
df["global_id"] = df["global_id"].astype(int)

results_df = results_df.merge(df, on="global_id", how="left")

In [23]:
# NOTE: Adaptyv results here!


In [24]:
results_df.to_csv("../data/12k_all_results.csv", index=False)